# JSON Basics — 10: JSON Best Practices

Capstone: combining all JSON lessons into production guidelines
and a complete reference for common JSON patterns.

Topics: schema registry for JSON, deduplication, large file handling,
production checklist, common pitfalls.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Schema Management



In [ ]:

import json as pyjson, pathlib

# Maintain schema versions in files (poor man's schema registry)
SCHEMA_REGISTRY = f"{DATA_DIR}/schema_registry"
pathlib.Path(SCHEMA_REGISTRY).mkdir(exist_ok=True)

v1_schema_json = {
    "version":1,"format":"json",
    "fields":{"order_id":"long","customer_id":"long","product":"string",
              "category":"string","price":"double","status":"string","order_date":"date"}
}
v2_schema_json = {
    "version":2,"format":"json",
    "fields":{"order_id":"long","customer_id":"long","product":"string",
              "category":"string","price":"double","discount":"double",
              "status":"string","order_date":"date","channel":"string"}
}
for v,s in [(1,v1_schema_json),(2,v2_schema_json)]:
    with open(f"{SCHEMA_REGISTRY}/v{v}.json","w") as f: pyjson.dump(s,f,indent=2)
print("Schema registry created:")
for v in [1,2]:
    schema_def = pyjson.loads(open(f"{SCHEMA_REGISTRY}/v{v}.json").read())
    print(f"  v{v}: {len(schema_def['fields'])} fields")


## Step 2 — Deduplication Pattern



In [ ]:

import json as pyjson, pathlib, random, datetime
random.seed(42)

# Create JSON with duplicates (common in at-least-once delivery)
CATS=["Electronics","Books","Clothing"]
rows=[{"order_id":i,"category":random.choice(CATS),
       "revenue":round(random.uniform(10,500),2),"status":"confirmed"}
      for i in range(1,5001)]
# Duplicate 10% of rows
dupes = random.sample(rows, 500)
all_rows = rows + dupes
random.shuffle(all_rows)

dup_path = f"{DATA_DIR}/with_dupes.json"
with open(dup_path,"w") as f: [f.write(pyjson.dumps(r)+"\n") for r in all_rows]
print(f"Total rows with dupes: {len(all_rows):,}")

df_raw = spark.read.json(dup_path)
print(f"Read: {df_raw.count():,} rows")

# Dedup by order_id (keep first occurrence)
df_dedup = df_raw.dropDuplicates(["order_id"])
print(f"After dedup: {df_dedup.count():,} rows")

# Dedup with window function (keep latest by some ordering)
from pyspark.sql.window import Window
w = Window.partitionBy("order_id").orderBy(F.col("status"))
df_dedup_wf = df_raw.withColumn("_rn", F.row_number().over(w)) \
                    .filter(F.col("_rn")==1).drop("_rn")
print(f"After window dedup: {df_dedup_wf.count():,} rows")


## Step 3 — Production Checklist



In [ ]:

print("""
JSON Production Checklist:
══════════════════════════════════════════════════════════

READING:
  ☐ Always define explicit schema (never inferSchema in production)
  ☐ Use PERMISSIVE mode + columnNameOfCorruptRecord
  ☐ Log and quarantine corrupt records
  ☐ Set encoding explicitly if source is not UTF-8
  ☐ Use input_file_name() to track source files

WRITING:
  ☐ Use gzip compression for JSON files at rest
  ☐ Write NDJSON (one object per line) — not pretty-printed
  ☐ Include all required fields (no silent omission of nulls)

PIPELINE:
  ☐ Convert JSON → Parquet as first step (before analytics)
  ☐ Validate row counts after conversion
  ☐ Implement checkpoint for incremental processing
  ☐ Deduplicate before writing to analytics zone
  ☐ Version your schemas

STREAMING:
  ☐ Always define explicit schema (required for streaming)
  ☐ Use from_json() for Kafka value deserialization
  ☐ Handle parse errors with PERMISSIVE mode
  ☐ Monitor for schema drift (new fields, type changes)

COMMON PITFALLS:
  ❌ Using multiLine=True on large files (disables parallelism)
  ❌ Using samplingRatio < 1.0 (misses types in tail)
  ❌ Storing analytics data as JSON (use Parquet/Delta/Iceberg)
  ❌ Not deduplicating at-least-once delivery streams
  ❌ Ignoring corrupt records without logging them
""")


## Step 4 — JSON Basics Recap



In [ ]:

print("""
JSON Basics — 10 notebooks summary:
  01 reading_json         — spark.read.json, multiLine, corrupt records, options
  02 writing_json         — compression, dates, single file, write modes
  03 schema_inference     — inferSchema cost, samplingRatio risk, explicit schema
  04 nested_json          — struct access, explode, from_json/to_json, get_json_object
  05 json_streaming       — file stream, Kafka JSON, from_json in streaming
  06 json_performance     — JSON vs Parquet benchmark, why JSON is slow
  07 json_schema_validation — PERMISSIVE + corrupt capture, business rules, quarantine
  08 json_rest_apis       — wrapped responses, pagination, API data normalization
  09 json_to_parquet      — landing zone pipeline, incremental, validation
  10 json_best_practices  — schema management, dedup, production checklist
""")
